# Lab 04 — CNNs, residual connections and depth

**Dataset:** [ylecun/mnist](https://huggingface.co/datasets/ylecun/mnist). We compare a small fully-connected model with a CNN and then test whether a residual block changes optimization behavior.

## The math behind the experiment

A convolution applies a small kernel across local image regions. In simplified notation,

$$
y_{i,j}=\sum_{u,v}K_{u,v}x_{i+u,j+v}+b.
$$

The same kernel is reused at many image locations. That is **parameter sharing**: the model can learn an edge detector once and use it across the image.

A residual block instead learns a correction to an existing representation:

$$
y=x+F(x).
$$

The shortcut gives gradients and information a direct path through the network, which helps very deep models train.

In [ ]:
!pip -q install datasets torch torchvision pandas matplotlib
from datasets import load_dataset
import numpy as np,pandas as pd,torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader
tr=load_dataset("ylecun/mnist",split="train[:4000]"); te=load_dataset("ylecun/mnist",split="test[:1000]")
def make(ds): return torch.tensor(np.array([np.array(x) for x in ds["image"]])/255.,dtype=torch.float32).unsqueeze(1),torch.tensor(ds["label"])
Xtr,ytr=make(tr); Xte,yte=make(te); print(Xtr.shape)

## Step 1 — Prediction

CNNs should need fewer parameters than a dense network for image structure because the same local filter is reused across positions. Expect competitive or better accuracy despite fewer parameters.

In [ ]:
class Dense(nn.Module):
 def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Flatten(),nn.Linear(784,64),nn.ReLU(),nn.Linear(64,10))
 def forward(self,x): return self.net(x)
class CNN(nn.Module):
 def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(32,10))
 def forward(self,x): return self.net(x)
def fit(m,epochs=3):
 opt=torch.optim.Adam(m.parameters(),lr=1e-3); loss=nn.CrossEntropyLoss(); dl=DataLoader(TensorDataset(Xtr,ytr),64,shuffle=True)
 for _ in range(epochs):
  for xb,yb in dl: opt.zero_grad(); loss(m(xb),yb).backward(); opt.step()
 with torch.no_grad(): acc=(m(Xte).argmax(1)==yte).float().mean().item()
 return sum(p.numel() for p in m.parameters()),acc
rows=[]
for n,m in [("dense",Dense()),("cnn",CNN())]: rows.append([n,*fit(m)])
pd.DataFrame(rows,columns=["model","parameters","test_accuracy"])

## Step 2 — Receptive-field intervention

Replace the CNN's local 3×3 kernels with a 1×1 convolution while keeping the training protocol fixed. Predict that the modified model should lose the spatial inductive bias.

In [ ]:
class CNN1x1(nn.Module):
 def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Conv2d(1,16,1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(16,32,1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(32,10))
 def forward(self,x): return self.net(x)
rows.append(["cnn_1x1",*fit(CNN1x1())]); pd.DataFrame(rows,columns=["model","parameters","test_accuracy"])

## Read the graph — accuracy must be considered with model size

The plot compares parameter count and test accuracy for the dense baseline, spatial CNN, and 1×1-convolution intervention. The 1×1 model removes most local spatial mixing, while the 3×3 model can learn patterns across neighboring pixels.

Do not conclude that one architecture is universally best from one short run. Check the parameter budget, training time, random seed, and variability before making a stronger claim.

In [ ]:
import matplotlib.pyplot as plt

model_results = pd.DataFrame(rows, columns=["model", "parameters", "test_accuracy"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(model_results["model"], model_results["parameters"])
axes[0].set_title("Model size")
axes[0].set_ylabel("Trainable parameters")
axes[0].tick_params(axis="x", rotation=15)
axes[0].grid(axis="y", alpha=.25)

axes[1].bar(model_results["model"], model_results["test_accuracy"])
axes[1].set_title("Held-out accuracy")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", alpha=.25)
fig.tight_layout()
plt.show()

## Step 3 — Does an identity skip help optimization?

Now isolate the residual connection itself. Both models use the same stem, two \(3\times3\) convolutions, pooling, classifier, optimizer, training examples, number of epochs, and parameter count. The only architectural change is whether the block adds its input back:

$$
\text{plain}(x)=\operatorname{ReLU}(F(x)),
\qquad
\text{residual}(x)=\operatorname{ReLU}(x+F(x)).
$$

For each seed, both models start from the same initial weights and see the same minibatch order. We repeat the comparison across three seeds and track training loss by epoch plus final held-out accuracy. This is a small teaching experiment, not enough evidence to claim a universal residual-network advantage.

In [ ]:
import torch.nn.functional as F

class ConvBlockModel(nn.Module):
    def __init__(self,residual=False):
        super().__init__()
        self.residual=residual
        self.stem=nn.Conv2d(1,16,3,padding=1)
        self.conv1=nn.Conv2d(16,16,3,padding=1)
        self.conv2=nn.Conv2d(16,16,3,padding=1)
        self.head=nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32,10)
        )
    def forward(self,x):
        x=F.relu(self.stem(x))
        skip=x
        x=F.relu(self.conv1(x))
        x=self.conv2(x)
        if self.residual:
            x=x+skip
        x=F.relu(x)
        return self.head(x)

def run_block_model(residual,seed,epochs=3):
    torch.manual_seed(seed)
    model=ConvBlockModel(residual=residual)
    n_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
    optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
    criterion=nn.CrossEntropyLoss()
    generator=torch.Generator().manual_seed(seed+100)
    loader=DataLoader(TensorDataset(Xtr,ytr),batch_size=64,shuffle=True,generator=generator)
    history=[]
    for epoch in range(1,epochs+1):
        model.train(); losses=[]
        for xb,yb in loader:
            optimizer.zero_grad()
            loss=criterion(model(xb),yb)
            loss.backward(); optimizer.step()
            losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            logits=model(Xte)
            test_loss=criterion(logits,yte).item()
            test_acc=(logits.argmax(1)==yte).float().mean().item()
        history.append({
            "seed":seed,
            "model":"residual" if residual else "plain",
            "epoch":epoch,
            "train_loss":float(np.mean(losses)),
            "test_loss":test_loss,
            "test_accuracy":test_acc,
            "parameters":n_params
        })
    return history

residual_rows=[]
for seed in [0,1,2]:
    # Resetting the seed gives both variants identical initialization and data order.
    residual_rows.extend(run_block_model(False,seed))
    residual_rows.extend(run_block_model(True,seed))
residual_results=pd.DataFrame(residual_rows)
param_check=residual_results.groupby("model")["parameters"].first()
assert param_check.nunique()==1, f"Parameter counts differ: {param_check.to_dict()}"
print("Trainable parameters (matched):",param_check.to_dict())
display(residual_results[residual_results["epoch"]==3][["seed","model","parameters","test_loss","test_accuracy"]])

In [ ]:
import matplotlib.pyplot as plt

fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for model_name in ["plain","residual"]:
    part=residual_results[residual_results["model"]==model_name]
    summary=part.groupby("epoch")["train_loss"].agg(["mean","std"]).reset_index()
    axes[0].errorbar(summary["epoch"],summary["mean"],yerr=summary["std"].fillna(0),
                     marker="o",capsize=4,label=model_name)
axes[0].set_title("Training loss by epoch (3 seeds)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Training cross-entropy")
axes[0].legend()
axes[0].grid(alpha=.25)

final=residual_results[residual_results["epoch"]==residual_results["epoch"].max()]
summary=final.groupby("model")["test_accuracy"].agg(["mean","std"]).reindex(["plain","residual"])
axes[1].bar(summary.index,summary["mean"],yerr=summary["std"].fillna(0),capsize=4)
axes[1].set_title("Final held-out accuracy (3 seeds)")
axes[1].set_ylabel("Accuracy (mean ± sample SD)")
axes[1].set_ylim(0,1)
axes[1].grid(axis="y",alpha=.25)
fig.tight_layout()
plt.show()

### Interpretation and answer key

The parameter-count assertion should pass because the residual connection adds no learned parameters. Compare the loss curves to see whether the skip connection changes optimization speed or stability; compare final accuracy only after checking the variation across seeds. If the residual model does not win in this short run, that does not disprove residual learning: the network is shallow, the dataset subset is small, and training budget matters.

**Research extension:** repeat with at least five seeds and a deeper matched pair. Report wall-clock time as well as loss and accuracy, and avoid attributing a difference to skip connections if the initialization, minibatch order, or training budget differs.

## Conclusion / answer key

Expected: the CNN should exploit local spatial structure more effectively than the dense baseline at comparable scale. The 1×1 intervention removes most local spatial mixing, so any drop is evidence for the convolutional inductive bias rather than merely “more layers.”

### Research extension
Compare residual vs plain blocks at equal depth and parameter count, measuring optimization curves and final accuracy.